# Lab 01: The Invisible Hole in Your Eye

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 1, *The Reality Generator*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-01.ipynb)

**What you will do:** measure the distance at which your own blind spot swallows a dot, turn that measurement into a visual angle, compare it with the eccentricity the book reports, and test a classical inpainting algorithm's version of "filling in the gap".

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 01 you drew (or used Figure 1.1's printed version of) a cross and a dot on paper, closed
one eye, fixated on the cross, and slowly brought the page towards your face until the dot
vanished completely into your blind spot. You then tried the line-completion and pattern-fill
variations, watching your brain paint a plausible line or pattern over the gap, and swapped
eyes to find the blind spot on the other side.

## Record your results

The book's "Full version" draws the cross and dot **15 cm apart**; the quick version printed
in Figure 1.1 uses **8 cm**. Type the separation you used and the distance from your face at
which the dot (or, with eyes swapped, the cross) vanished, for each eye you tested. The angle
follows the book's own worked example (`15 cm ÷ tan 15° ≈ 56 cm`): `angle = atan(separation /
distance)`.

In [ ]:
# example data: replace with your own measurements, in centimetres
trials = pd.DataFrame({
    "eye_tested": ["right (dot vanishes, Steps 1-7)", "left (cross vanishes, Variation 3)"],
    "separation_cm": [15.0, 15.0],   # example data: how far apart you drew the cross and dot
    "distance_cm": [54.0, 58.0],     # example data: distance from your face when it vanished
})
trials["eccentricity_deg"] = np.degrees(np.arctan(trials["separation_cm"] / trials["distance_cm"]))
display(trials)

trials.plot.bar(x="eye_tested", y="eccentricity_deg", legend=False,
                 color=["steelblue", "firebrick"], figsize=(6, 3))
plt.axhline(15, color="grey", linestyle="--", label="book's ~15°")
plt.ylabel("computed eccentricity (degrees)"); plt.legend(); plt.tight_layout(); plt.show()

## Compare

Lab 01 explains that the blind spot sits "about 15 degrees to the side of your fixation
point", and works through an example using its own numbers: "a dot 15 cm from the cross falls
on it only when the page is roughly 15 cm ÷ tan(15°) ≈ 56 cm from your eye." Compare your own
computed angle above with that ~15°.

In [ ]:
for _, row in trials.iterrows():
    diff = row["eccentricity_deg"] - 15
    print(f"{row['eye_tested']}: {row['eccentricity_deg']:.1f}°  "
          f"(book: ~15°, difference {diff:+.1f}°)")
print()
print("The book notes that the exact vanishing distance 'varies a little from person to")
print("person', so a difference of a few degrees is expected. A much larger gap more likely")
print("means your fixation drifted towards the dot, or the page moved unevenly, than a genuine")
print("difference in your eye's anatomy.")

## The AI side

Your brain fills the blind spot with content generated from its model of the surrounding
scene -- the same operation photo-editing software calls **inpainting**, as the AI Connection
section notes for tools like Photoshop's Content-Aware Fill. Below, a classical (non-learned)
inpainting algorithm built into OpenCV fills a masked disc in a pattern we made ourselves, so
we know exactly what the "true" pixels were -- something you can never check for your own
blind spot, because you have no independent way to see what was really there.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import cv2
from PIL import Image, ImageDraw

# Build a scene with a known ground truth: blue stripes over a warm background
scene = Image.new("RGB", (256, 256), (235, 225, 205))
draw = ImageDraw.Draw(scene)
for y in range(0, 256, 16):
    draw.line([(0, y), (256, y)], fill=(70, 110, 160), width=6)
scene_arr = np.array(scene)

# Punch a hole where the "blind spot" falls, and tell OpenCV where the hole is
mask = np.zeros((256, 256), dtype=np.uint8)
cv2.circle(mask, (150, 130), 35, 255, -1)
damaged = scene_arr.copy()
damaged[mask == 255] = (0, 0, 0)

filled = cv2.inpaint(damaged, mask, inpaintRadius=5, flags=cv2.INPAINT_TELEA)
error = np.abs(filled.astype(int) - scene_arr.astype(int))[mask == 255].mean()

fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
for ax, img, title in zip(axes, [scene_arr, damaged, filled],
                          ["Ground truth", "Damaged (the 'blind spot')", "Inpainted"]):
    ax.imshow(img); ax.set_title(title, fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()
print(f"Mean pixel error inside the filled disc: {error:.1f} (0-255 scale, 0 = perfect)")

On a regular striped pattern the classical fill is nearly perfect, because the missing content
is highly predictable from its surroundings -- exactly the case your brain handles so well at
the real blind spot. The analogy breaks down where the surroundings stop being predictable:
put a one-off object in the middle of the stripes and both this algorithm and your own visual
system will confidently paint over it with plausible stripes instead, each unaware that
anything went missing. Modern AI inpainting (Stable Diffusion and similar) uses a learned
model of whole scenes rather than just local texture, which is closer to what your brain does
-- but it can hallucinate just as confidently and just as undetectably.

## Questions to think about

1. If you had used the Figure 1.1 quick version (8 cm apart) instead of the 15 cm full version, roughly what distance would make the dot vanish at the same ~15° angle? Work it out from the formula and check it against the book's own 8 cm example.
2. In Variation 1 (line completion) the brain extended a straight line across the blind spot; in Variation 2 (pattern fill) it extended a pattern of dots. What do these two results tell you about whether the fill-in content is a fixed default (e.g. always plain background) or specific to the local context?
3. The inpainting demo above filled a regular striped pattern almost perfectly. Predict what would happen to the mean pixel error if the pattern were random coloured blotches instead of even stripes, and explain why using the same "prediction from context" logic that explains blind-spot filling.
4. The book argues that having two eyes prevents the blind spot from ever appearing as a black hole. Using your own two eccentricity measurements above (one per eye), explain why the two eyes' blind spots do not overlap in space.

## Challenge

Test whether the *content* of the fill-in is arbitrary or tied to what surrounds the blind
spot. Draw two versions of the Variation 2 pattern-fill card: one with red dots around the
(empty) blind-spot circle, another with blue dots. Find your blind spot with each card and note,
in your own words, what colour (if any) seems to fill the gap.

In [ ]:
# Example data: replace with your own reports after trying the red-dot and blue-dot cards.
my_reports = pd.DataFrame({
    "surrounding_dot_colour": ["red", "blue"],
    "colour_seen_in_gap": ["red-ish", "blue-ish"],   # example data: what you actually saw
    "confidence_1to5": [4, 3],                        # example data
})
display(my_reports)
print("If the colour you saw in the gap tends to match the surrounding dots, that supports the")
print("book's claim (citing Ramachandran & Gregory, 1991) that filling-in uses local context")
print("rather than a fixed default.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-01.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")